In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.model_selection import learning_curve
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.feature_selection import RFE 
from sklearn.svm import SVC
import random
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

               hotel  is_canceled  lead_time  arrival_date_year  \
0       Resort Hotel            0        342               2015   
1       Resort Hotel            0        737               2015   
2       Resort Hotel            0          7               2015   
3       Resort Hotel            0         13               2015   
4       Resort Hotel            0         14               2015   
...              ...          ...        ...                ...   
119385    City Hotel            0         23               2017   
119386    City Hotel            0        102               2017   
119387    City Hotel            0         34               2017   
119388    City Hotel            0        109               2017   
119389    City Hotel            0        205               2017   

       arrival_date_month  arrival_date_week_number  \
0                    July                        27   
1                    July                        27   
2                    July     

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import GradientBoostingClassifier
import pickle

# Load the CSV file into a DataFrame
df = pd.read_csv("/Users/kevinleungch421/Desktop/Hotel-Booking-Cancellation-Analysis/data/hotel_bookings.csv")

print("Report of data cleaning:\n")

# 1. Randomly sample 40,000 data points
random_seed = 42
df = df.sample(n=40000, random_state=random_seed, replace=False)
print(f"1. after the random sample, the number of rows is: {len(df)}\n")

# 2. Create new feature 'season' based on 'arrival_date_month'
month_to_season = {
    'January': 'Winter', 'February': 'Winter', 'March': 'Spring', 'April': 'Spring',
    'May': 'Spring', 'June': 'Summer', 'July': 'Summer', 'August': 'Summer',
    'September': 'Autumn', 'October': 'Autumn', 'November': 'Autumn', 'December': 'Winter'
}
df['season'] = df['arrival_date_month'].replace(month_to_season)
print(f"2. 'season' unique values: {df['season'].unique()}\n")

# 3. Delete meaningless and complex features
useless_col = [
    'assigned_room_type',
    'country',
    'arrival_date_week_number',
    'arrival_date_day_of_month',
    'agent',
    'company',
    'reservation_status_date',
    'reservation_status',
    'arrival_date_month'
]
df = df.drop(columns=[col for col in useless_col if col in df.columns])
print(f"3. after dropping useless columns, number of columns is: {len(df.columns)}\n")

# 4. Clean null values in 'children'
df['children'] = df['children'].fillna(0)
print(f"4. 'children' has any nulls? {df['children'].isnull().any()}\n")

# 5. Correct the data type in 'children'
df['children'] = df['children'].astype('int64')
print(f"5. 'children' dtype is now: {df['children'].dtype}\n")

# 6. Merge 'Undefined' into 'SC' category in 'meal'
df['meal'].replace('Undefined', 'SC', inplace=True)
print(f"6. 'meal' unique values: {df['meal'].unique()}\n")

# 7. Handle outliers in specified numeric features
print("7. handling outliers")
numeric_features_outlier = [
    'lead_time',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adr',
    'total_of_special_requests'
]
for feature in numeric_features_outlier:
    mean = df[feature].mean()
    std = df[feature].std()
    upper = mean + std
    lower = mean - std
    df[feature] = np.where(df[feature] > upper, upper, df[feature])
    df[feature] = np.where(df[feature] < lower, lower, df[feature])
print("Outliers handled.\n")

# One-hot encode categorical columns (dropping first to avoid multicollinearity)
columns_to_encode = [
    'hotel', 'arrival_date_year', 'is_repeated_guest', 'meal',
    'market_segment', 'distribution_channel', 'reserved_room_type',
    'deposit_type', 'customer_type', 'season'
]
df_encoded = pd.get_dummies(df,
                            columns=columns_to_encode,
                            prefix="dummy",
                            drop_first=True)

# Convert all dummy columns to int
for col in df_encoded.columns:
    if col.startswith('dummy_'):
        df_encoded[col] = df_encoded[col].astype(int)

# Split into features and target
y = df_encoded['is_canceled']
X = df_encoded.drop('is_canceled', axis=1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1)

# Scale features
scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_train.columns
)

print("\nShape of the Train/Test Split:\n")
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)
print("y_train       :", y_train.shape)
print("y_test        :", y_test.shape)

from sklearn.preprocessing import MinMaxScaler
import pickle

# ... after train/test split ...
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# save the scaler
with open("minmax_scaler.pkl", "wb") as f_scaler:
    pickle.dump(scaler, f_scaler)

# now train your model
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=500, max_depth=7, random_state=1)
gb.fit(X_train_scaled, y_train)

# save the model
with open("gradient_boosting_hotel_cancellation.pkl", "wb") as f_model:
    pickle.dump(gb, f_model)
    
# Initialize and train Gradient Boosting model
gb = GradientBoostingClassifier(n_estimators=500, max_depth=7, random_state=1)
gb.fit(X_train_scaled, y_train)

# Save the trained model using pickle
model_filename = "gradient_boosting_hotel_cancellation.pkl"
with open(model_filename, "wb") as f:
    pickle.dump(gb, f)

print(f"\nModel trained and saved to {model_filename}")

Report of data cleaning:

1. after the random sample, the number of rows is: 40000

2. 'season' unique values: ['Winter' 'Summer' 'Spring' 'Autumn']

3. after dropping useless columns, number of columns is: 24

4. 'children' has any nulls? False

5. 'children' dtype is now: int64

6. 'meal' unique values: ['BB' 'SC' 'HB' 'FB']

7. handling outliers
Outliers handled.


Shape of the Train/Test Split:

X_train_scaled: (32000, 47)
X_test_scaled : (8000, 47)
y_train       : (32000,)
y_test        : (8000,)

Model trained and saved to gradient_boosting_hotel_cancellation.pkl


In [5]:
from flask import Flask, request, jsonify
import pandas as pd
import pickle

# 1. create Flask app instance
app = Flask(__name__)

# 2. load trained model and scaler (make sure these .pkl files are present)
with open("gradient_boosting_hotel_cancellation.pkl", "rb") as f:
    model = pickle.load(f)

with open("minmax_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# 3. list of all features in the order expected by the model
feature_columns = [
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "dummy_Resort Hotel",
    "dummy_2016",
    "dummy_2017",
    "dummy_1",
    "dummy_FB",
    "dummy_HB",
    "dummy_SC",
    "dummy_Complementary",
    "dummy_Corporate",
    "dummy_Direct",
    "dummy_Groups",
    "dummy_Offline TA/TO",
    "dummy_Online TA",
    "dummy_GDS",
    "dummy_TA/TO",
    "dummy_Undefined",
    "dummy_B",
    "dummy_C",
    "dummy_D",
    "dummy_E",
    "dummy_F",
    "dummy_G",
    "dummy_H",
    "dummy_L",
    "dummy_P",
    "dummy_Non Refund",
    "dummy_Refundable",
    "dummy_Group",
    "dummy_Transient",
    "dummy_Transient-Party",
    "dummy_Spring",
    "dummy_Summer",
    "dummy_Winter"
]

# 4. home route
@app.route("/", methods=["GET"])
def index():
    return """
    <h1>Hotel Booking Cancellation Predictor</h1>
    <p>POST to <code>/predict</code> with a JSON body containing all model features.</p>
    """

# 5. prediction endpoint
@app.route("/predict", methods=["POST"])
def predict():
    # parse incoming JSON into DataFrame
    data = request.get_json(force=True)
    df_input = pd.DataFrame([data])

    # ensure all expected features are present; missing ones set to 0
    for col in feature_columns:
        if col not in df_input.columns:
            df_input[col] = 0

    # subset & reorder
    X = df_input[feature_columns]

    # scale
    X_scaled = scaler.transform(X)

    # predict: 0 = not canceled, 1 = canceled
    pred = model.predict(X_scaled)[0]

    return jsonify({
        "prediction": int(pred),
        "explanation": "1 = booking will be canceled, 0 = booking will not be canceled"
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=True)

FileNotFoundError: [Errno 2] No such file or directory: 'minmax_scaler.pkl'

In [ ]:
# train_model.py

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import GradientBoostingClassifier
import pickle

# 1. Load raw data
df = pd.read_csv("/Users/kevinleungch421/Desktop/Hotel-Booking-Cancellation-Analysis/data/hotel_bookings.csv")

# 2. Random sample for efficiency
df = df.sample(n=40000, random_state=42)

# 3. Map months → seasons
month_to_season = {
    'January':'Winter','February':'Winter','March':'Spring','April':'Spring',
    'May':'Spring','June':'Summer','July':'Summer','August':'Summer',
    'September':'Autumn','October':'Autumn','November':'Autumn','December':'Winter'
}
df['season'] = df['arrival_date_month'].map(month_to_season)

# 4. Drop useless columns
drop_cols = [
    'assigned_room_type','country','arrival_date_week_number',
    'arrival_date_day_of_month','agent','company',
    'reservation_status_date','reservation_status','arrival_date_month'
]
df.drop(columns=[c for c in drop_cols if c in df], inplace=True)

# 5. Fill and cast children
df['children'].fillna(0, inplace=True)
df['children'] = df['children'].astype(int)

# 6. Merge 'Undefined' → 'SC' in meal
df['meal'].replace('Undefined', 'SC', inplace=True)

# 7. Cap outliers at ±1 std
numeric_cols = [
    'lead_time','stays_in_weekend_nights','stays_in_week_nights',
    'adr','total_of_special_requests'
]
for c in numeric_cols:
    μ, σ = df[c].mean(), df[c].std()
    df[c] = np.clip(df[c], μ - σ, μ + σ)

# 8. One-hot encode (drop_first=True)
to_encode = [
    'hotel','arrival_date_year','is_repeated_guest','meal',
    'market_segment','distribution_channel','reserved_room_type',
    'deposit_type','customer_type','season'
]
df_encoded = pd.get_dummies(df, columns=to_encode, prefix='dummy', drop_first=True)

# 9. Split X / y
y = df_encoded.pop('is_canceled')
X = df_encoded

# 10. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

# 11. Fit scaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 12. Save scaler
with open("minmax_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# 13. Train model
gb = GradientBoostingClassifier(n_estimators=500, max_depth=7, random_state=1)
gb.fit(X_train_scaled, y_train)

# 14. Save model
with open("gradient_boosting_hotel_cancellation.pkl", "wb") as f:
    pickle.dump(gb, f)

print("Training complete. Saved minmax_scaler.pkl & gradient_boosting_hotel_cancellation.pkl")